**Week 12 — LDA, QDA, Gaussian Naive Bayes และ KNN ธุรกิจ — ทำนายลูกค้าตอบรับโปรโมชัน/เลิกซื้อ**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:** `cust = pd.read_csv('biz_customers.csv')` (1203 แถว แถวละ 1 คน)
- **การแบ่งชุดข้อมูล:** train = 960 แถวแรก (`.iloc[:960]`), test = 243 แถวที่เหลือ (`.iloc[960:]`) โดยไม่สลับข้อมูล
- **การจำแนกประเภท:** คลาสบวก = 1 ใช้ค่าเริ่มต้นของ scikit-learn ทุกอย่างยกเว้นที่ระบุเฉพาะข้อ
- **การปรับสเกลตัวแปร:** ข้อที่ระบุให้ standardize ให้ fit `StandardScaler` บนชุด train แล้วนำไป transform ทั้งชุด train และ test
- **KNN:** ใช้ระยะทางแบบยุคลิดและไม่ถ่วงน้ำหนักเพื่อนบ้าน (uniform weights)

### รายละเอียดไฟล์ข้อมูล `biz_customers.csv` (1203 แถว)

- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)
- `orders_90d`: จำนวนออเดอร์ใน 90 วันล่าสุด
- `churn`: ลูกค้าเลิกซื้อหรือไม่ (1 = เลิกซื้อ, 0 = ยังซื้ออยู่)
- `responded`: ลูกค้าตอบรับโปรโมชันหรือไม่ (1 = ตอบรับ, 0 = ไม่ตอบรับ)

**โจทย์ข้อ 12.1:** ฝึกใช้ข้อมูล train (960 แถวแรกของ `biz_customers.csv`) ตัวแปรตาม = `responded` ตัวทำนาย (ตามลำดับ) = `email_opens`, `web_visits`, `income_k`, `age` fit LDA **ค่าเฉลี่ยของ `web_visits` ของคลาส 1 ที่ LDA ประมาณได้ (`means_`)** = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 1. เตรียมข้อมูล train (960 แถวแรก)
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
features = ['email_opens', 'web_visits', 'income_k', 'age']

# 2. Fit LDA
lda = LinearDiscriminantAnalysis()
lda.fit(train[features], train['responded'])
print("คลาสที่โมเดลเรียนรู้:", lda.classes_)
print("เมทริกซ์ค่าเฉลี่ยของแต่ละคลาส (means_):\n", lda.means_)

# 3. ค่าเฉลี่ยของ web_visits (คอลัมน์ index 1) ของคลาส 1 (แถว index 1)
mean_web_class1 = lda.means_[1, 1]
print(f"ค่าเฉลี่ย web_visits ของคลาส 1: {mean_web_class1:.3f}")
print(f"คำตอบข้อ 12.1: {mean_web_class1:.3f}")


คลาสที่โมเดลเรียนรู้: [0 1]
เมทริกซ์ค่าเฉลี่ยของแต่ละคลาส (means_):
 [[ 4.93113772  7.69311377 30.0498503  39.59730539]
 [ 5.9109589   9.13013699 31.33356164 36.53424658]]
ค่าเฉลี่ย web_visits ของคลาส 1: 9.130
คำตอบข้อ 12.1: 9.130


**โจทย์ข้อ 12.2:** ฝึกใช้ไฟล์ `biz_customers.csv` ตัวแปรตาม = `loyalty_member` (1 = เป็นสมาชิกสะสมแต้ม) ตัวทำนาย = `web_visits`, `tenure_months`, `orders_90d`, `email_opens` fit LDA ด้วยข้อมูล train (960 แถวแรกของไฟล์) แล้ววัดผลกับชุด test (243 แถวที่เหลือ) ด้วย `predict` **ค่า F1 score ของชุด test (คลาสบวก = 1 คือเป็นสมาชิก)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import f1_score

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['web_visits', 'tenure_months', 'orders_90d', 'email_opens']

# 2. Fit LDA บนชุด train
lda = LinearDiscriminantAnalysis()
lda.fit(train[features], train['loyalty_member'])

# 3. ทำนายชุด test และคำนวณ F1 score
test_pred = lda.predict(test[features])
f1 = f1_score(test['loyalty_member'], test_pred)
print(f"F1 score บนชุด test: {f1:.4f}")
print(f"คำตอบข้อ 12.2: {f1:.4f}")


F1 score บนชุด test: 0.6623
คำตอบข้อ 12.2: 0.6623


**โจทย์ข้อ 12.3:** กรณีศึกษาร้านขายเครื่องใช้ไฟฟ้าออนไลน์ใช้ข้อมูล train (960 แถวแรก) สร้าง Quadratic Discriminant Analysis (QDA) ทำนายการตอบรับโปรโมชันจากจำนวนอีเมลโปรโมชันที่เปิดอ่าน, จำนวนครั้งที่เข้าเว็บ/แอป, รายได้ และอายุ ลูกค้ารายใหม่ที่ไม่อยู่ในไฟล์มีจำนวนอีเมลโปรโมชันที่เปิดอ่าน 4 ฉบับ, จำนวนครั้งที่เข้าเว็บ/แอป 10 ครั้ง, รายได้ 41.8 พันบาท/เดือน, อายุ 28 ปี **ความน่าจะเป็นที่ QDA ให้กับคลาส "ตอบรับโปรโมชัน" (`responded = 1`) ของลูกค้ารายนี้ (`predict_proba`)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# 1. Fit QDA บน train
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
features = ['email_opens', 'web_visits', 'income_k', 'age']
qda = QuadraticDiscriminantAnalysis()
qda.fit(train[features], train['responded'])

# 2. กำหนดข้อมูลลูกค้ารายใหม่
new_customer = pd.DataFrame([{'email_opens': 4, 'web_visits': 10, 'income_k': 41.8, 'age': 28}])

# 3. คำนวณความน่าจะเป็นของคลาส 1
prob_class1 = qda.predict_proba(new_customer)[0, 1]
print(f"ความน่าจะเป็นที่จะตอบรับโปรโมชัน: {prob_class1:.4f}")
print(f"คำตอบข้อ 12.3: {prob_class1:.4f}")


ความน่าจะเป็นที่จะตอบรับโปรโมชัน: 0.2735
คำตอบข้อ 12.3: 0.2735


**โจทย์ข้อ 12.4:** กรณีศึกษาทีมการตลาดของร้านเครื่องสำอางต้องการส่งโปรโมชันให้ลูกค้าที่จะตอบรับให้ได้มากที่สุด จึงลองใช้ Gaussian Naive Bayes ทำนายการตอบรับโปรโมชันจากจำนวนอีเมลโปรโมชันที่เปิดอ่าน, จำนวนครั้งที่เข้าเว็บ/แอป และอายุ fit ด้วยข้อมูล train (960 แถวแรก) แล้วทำนายลูกค้าชุด test (243 แถวที่เหลือ) ด้วย `predict` และค่าเริ่มต้นของ scikit-learn **สัดส่วนของลูกค้าที่ตอบรับจริงในชุด test ที่โมเดลจับได้ (recall, คลาสบวก = 1)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['email_opens', 'web_visits', 'age']

# 2. Fit Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(train[features], train['responded'])

# 3. ทำนายชุด test และคำนวณ Recall
test_pred = gnb.predict(test[features])
recall = recall_score(test['responded'], test_pred)
print(f"Recall บนชุด test: {recall:.4f}")
print(f"คำตอบข้อ 12.4: {recall:.4f}")


Recall บนชุด test: 0.2025
คำตอบข้อ 12.4: 0.2025


**โจทย์ข้อ 12.5:** กรณีศึกษาทีมวิเคราะห์ข้อมูลของธนาคารเปรียบเทียบ LDA กับ QDA ที่ทำนายการตอบรับโปรโมชันจากจำนวนอีเมลโปรโมชันที่เปิดอ่าน, รายได้ และอายุ โดย fit ทั้งสองโมเดลด้วยข้อมูล train (960 แถวแรก) แล้วทำนายลูกค้าชุด test (243 แถวที่เหลือ) ด้วย `predict` **มีลูกค้าในชุด test กี่คนที่ LDA กับ QDA ทำนายคลาสไม่ตรงกัน** = ? คน (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['email_opens', 'income_k', 'age']

# 2. Fit ทั้ง LDA และ QDA
lda = LinearDiscriminantAnalysis().fit(train[features], train['responded'])
qda = QuadraticDiscriminantAnalysis().fit(train[features], train['responded'])

# 3. ทำนายชุด test และนับจำนวนที่ไม่ตรงกัน
pred_lda = lda.predict(test[features])
pred_qda = qda.predict(test[features])
mismatch_count = int(np.sum(pred_lda != pred_qda))
print(f"จำนวนลูกค้าที่ LDA และ QDA ทำนายไม่ตรงกัน: {mismatch_count} คน")
print(f"คำตอบข้อ 12.5: {mismatch_count}")


จำนวนลูกค้าที่ LDA และ QDA ทำนายไม่ตรงกัน: 9 คน
คำตอบข้อ 12.5: 9


**โจทย์ข้อ 12.6:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `web_visits`, `income_k`, `spend_k` standardize ตัวทำนายด้วยค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของ train (`StandardScaler`) แล้ว fit KNN ที่ $K = 21$ ด้วย train (960 แถวแรกของ `biz_customers.csv`) **ความน่าจะเป็นของคลาส 1 ที่ `predict_proba` ให้ลูกค้าแต่ละคนในชุด test (243 แถวที่เหลือ) นำมาเฉลี่ยกันทั้ง 243 คน** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. แบ่ง train / test และ standardize ด้วย StandardScaler
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['email_opens', 'web_visits', 'income_k', 'spend_k']

scaler = StandardScaler()
X_train_std = scaler.fit_transform(train[features])
X_test_std = scaler.transform(test[features])

# 2. Fit KNN (k=21)
knn = KNeighborsClassifier(n_neighbors=21)
knn.fit(X_train_std, train['responded'])

# 3. คำนวณความน่าจะเป็นของคลาส 1 บนชุด test และหาค่าเฉลี่ย
probs_test = knn.predict_proba(X_test_std)[:, 1]
mean_prob = np.mean(probs_test)
print(f"ค่าเฉลี่ยความน่าจะเป็นของคลาส 1: {mean_prob:.4f}")
print(f"คำตอบข้อ 12.6: {mean_prob:.4f}")


ค่าเฉลี่ยความน่าจะเป็นของคลาส 1: 0.3000
คำตอบข้อ 12.6: 0.3000


**โจทย์ข้อ 12.7:** กรณีศึกษานักวิเคราะห์ข้อมูลของห้างสรรพสินค้าสงสัยว่าหน่วยของตัวแปรมีผลต่อ KNN หรือไม่ จึง fit KNN ที่ $K = 9$ สองแบบเพื่อทำนายการตอบรับโปรโมชันจากจำนวนอีเมลโปรโมชันที่เปิดอ่าน, รายได้, อายุ และยอดซื้อ ด้วย train (960 แถวแรก) แบบที่ 1 ใช้ค่าดิบ แบบที่ 2 standardize ตัวทำนายด้วยค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของ train (`StandardScaler`) ก่อน **มีลูกค้าในชุด test (243 แถวที่เหลือ) กี่คนที่สองแบบทำนายคลาสไม่ตรงกัน** = ? คน (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. แบ่งชุดข้อมูล
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['email_opens', 'income_k', 'age', 'spend_k']

# 2. แบบที่ 1: ใช้ค่าดิบ
knn_raw = KNeighborsClassifier(n_neighbors=9).fit(train[features], train['responded'])
pred_raw = knn_raw.predict(test[features])

# 3. แบบที่ 2: Standardize
scaler = StandardScaler()
X_train_std = scaler.fit_transform(train[features])
X_test_std = scaler.transform(test[features])
knn_std = KNeighborsClassifier(n_neighbors=9).fit(X_train_std, train['responded'])
pred_std = knn_std.predict(X_test_std)

# 4. นับจำนวนลูกค้าที่ทำนายไม่ตรงกัน
mismatches = int(np.sum(pred_raw != pred_std))
print(f"จำนวนลูกค้าที่ผลทำนายไม่ตรงกัน: {mismatches} คน")
print(f"คำตอบข้อ 12.7: {mismatches}")


จำนวนลูกค้าที่ผลทำนายไม่ตรงกัน: 42 คน
คำตอบข้อ 12.7: 42


**โจทย์ข้อ 12.8:** กรณีศึกษาธุรกิจสมัครสมาชิกกล่องสินค้ารายเดือนใช้ LDA ทำนายการเลิกซื้อจากระยะเวลาที่เป็นลูกค้า, จำนวนครั้งที่เข้าเว็บ/แอป และอายุ fit ด้วยข้อมูล train (960 แถวแรก) แต่ predict ปกติจับลูกค้าที่จะเลิกซื้อได้น้อยเกินไป จึงเปลี่ยนเกณฑ์เป็นจัดว่าเสี่ยงเลิกซื้อ (`churn = 1`) เมื่อความน่าจะเป็นของคลาส 1 จาก `predict_proba` $\ge 0.12$ **ค่า recall ของชุด test (243 แถวที่เหลือ, คลาสบวก = 1 คือเลิกซื้อ)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import recall_score

# 1. แบ่งชุดข้อมูล
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['tenure_months', 'web_visits', 'age']

# 2. Fit LDA บน train
lda = LinearDiscriminantAnalysis().fit(train[features], train['churn'])

# 3. ทำนายความน่าจะเป็นและจัดคลาสด้วย threshold = 0.12
prob_test = lda.predict_proba(test[features])[:, 1]
pred_test = (prob_test >= 0.12).astype(int)

# 4. คำนวณ Recall
recall = recall_score(test['churn'], pred_test)
print(f"Recall ที่ threshold 0.12: {recall:.4f}")
print(f"คำตอบข้อ 12.8: {recall:.4f}")


Recall ที่ threshold 0.12: 0.6071
คำตอบข้อ 12.8: 0.6071


**โจทย์ข้อ 12.9:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `income_k`, `age`, `spend_k` standardize ตัวทำนายด้วยค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของ train (`StandardScaler`) แล้ว fit KNN ด้วย train (960 แถวแรกของ `biz_customers.csv`) ลอง $K = 1, 3, 5, \dots, 25$ (เลขคี่ ทั้งหมด 13 ค่า) **ค่า F1 score ของชุด test (243 แถวที่เหลือ, คลาสบวก = 1) ที่สูงสุดในบรรดา 13 ค่า $K$ นี้** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

# 1. เตรียมข้อมูลและ Standardization
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['email_opens', 'income_k', 'age', 'spend_k']

scaler = StandardScaler()
X_train_std = scaler.fit_transform(train[features])
X_test_std = scaler.transform(test[features])

# 2. ทดลอง K = 1, 3, ..., 25
k_values = list(range(1, 26, 2))
best_f1 = -1.0
best_k = None

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_std, train['responded'])
    pred_k = knn.predict(X_test_std)
    f1_k = f1_score(test['responded'], pred_k)
    print(f"K = {k:2d}: F1 score = {f1_k:.4f}")
    if f1_k > best_f1:
        best_f1 = f1_k
        best_k = k

print(f"\nค่า F1 สูงสุด: {best_f1:.4f} (ที่ K = {best_k})")
print(f"คำตอบข้อ 12.9: {best_f1:.4f}")


K =  1: F1 score = 0.3973
K =  3: F1 score = 0.3306
K =  5: F1 score = 0.3306
K =  7: F1 score = 0.2807
K =  9: F1 score = 0.2752
K = 11: F1 score = 0.2476
K = 13: F1 score = 0.1837
K = 15: F1 score = 0.1895
K = 17: F1 score = 0.1702
K = 19: F1 score = 0.1702
K = 21: F1 score = 0.1075
K = 23: F1 score = 0.1075
K = 25: F1 score = 0.1333

ค่า F1 สูงสุด: 0.3973 (ที่ K = 1)
คำตอบข้อ 12.9: 0.3973


**โจทย์ข้อ 12.10:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `web_visits`, `income_k`, `age` fit QDA ด้วยข้อมูล train (960 แถวแรกของ `biz_customers.csv`) **ค่า AUC ของชุด test (243 แถวที่เหลือ) โดยใช้ความน่าจะเป็นของคลาส 1 จาก `predict_proba` (คลาสบวก = 1)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import roc_auc_score

# 1. แบ่งชุดข้อมูล
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['email_opens', 'web_visits', 'income_k', 'age']

# 2. Fit QDA บน train
qda = QuadraticDiscriminantAnalysis().fit(train[features], train['responded'])

# 3. คำนวณ ROC AUC บนชุด test ด้วย predict_proba ของคลาส 1
probs_test = qda.predict_proba(test[features])[:, 1]
auc_score = roc_auc_score(test['responded'], probs_test)
print(f"ROC AUC Score บนชุด test: {auc_score:.4f}")
print(f"คำตอบข้อ 12.10: {auc_score:.4f}")


ROC AUC Score บนชุด test: 0.6907
คำตอบข้อ 12.10: 0.6907


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 12

---

### 1. การจำแนกประเภทแบบก่อกำเนิด (Generative Classifiers)

แทนที่จะประมาณ $P(Y=k|X)$ โดยตรง (เหมือน Logistic Regression) อัลกอริทึมกลุ่มนี้จะจำลองการแจกแจงร่วมผ่านกฎของเบย์ส (Bayes' Theorem):
$$P(Y = k \mid X = x) = \frac{\pi_k f_k(x)}{\sum_{l=1}^K \pi_l f_l(x)}$$
โดย $\pi_k$ คือความน่าจะเป็นก่อนหน้า (Prior probability) และ $f_k(x)$ คือฟังก์ชันความหนาแน่นความน่าจะเป็นแบบ Multivariate Normal ในแต่ละคลาส

---

### 2. เปรียบเทียบ LDA, QDA และ Gaussian Naive Bayes

| โมเดล | สมมติฐานเรื่อง Covariance Matrix ($\mathbf{\Sigma}_k$) | ขอบเขตการตัดสินใจ (Decision Boundary) | จำนวนพารามิเตอร์ที่ต้องประมาณ |
| :---: | :---: | :---: | :---: |
| **LDA** | ทุกคลาสใช้ Covariance Matrix เดียวกัน ($\mathbf{\Sigma}_k = \mathbf{\Sigma}$) | เส้นตรง / ระนาบ (Linear) | ปานกลาง (Low Variance, High Bias) |
| **QDA** | แต่ละคลาสมี Covariance Matrix เป็นของตนเอง ($\mathbf{\Sigma}_k$) | เส้นโค้งกำลังสอง (Quadratic) | สูง (High Variance, Low Bias) |
| **Gaussian NB** | ตัวแปรเป็นอิสระต่อกันอย่างสมบูรณ์ (Diagonal Covariance) | แกนขนาน (Non-linear) | ต่ำที่สุด (เหมาะกับ High Dimensional) |

---

### 3. K-Nearest Neighbors Classifier (KNN) และการปรับสเกลข้อมูล

- **ความไวต่อหน่วยวัด (Scale Sensitivity):** KNN คำนวณความใกล้เคียงด้วยระยะทาง Euclidean:
  $$d(x, z) = \sqrt{\sum_{j=1}^p (x_j - z_j)^2}$$
  ตัวแปรที่มีช่วงค่ากว้าง (เช่น `income_k` หรือ `spend_k`) จะครอบงำระยะทางทั้งหมด ดังนั้นจึงจำเป็นต้องทำ **Standardization (`StandardScaler`)** เสมอ เพื่อให้ทุกตัวแปรมีน้ำหนักเท่าเทียมกัน
- **Hyperparameter $K$:** ค่า $K$ ควบคุม Bias-Variance Tradeoff โดย $K=1$ มีความยืดหยุ่นสูงสุด แต่เสี่ยงต่อการ overfit สูง

---

### 4. การประเมินผลและการปรับ Decision Threshold

- **ROC AUC (Area Under the ROC Curve):** วัดประสิทธิภาพการจัดลำดับคะแนนความน่าจะเป็นของโมเดล โดยไม่ขึ้นกับเกณฑ์ threshold ที่เลือก ค่าสุ่มเดาคือ 0.5 และสมบูรณ์แบบคือ 1.0
- **Cost-sensitive / Imbalanced Thresholding:** ในกรณีที่ต้องการจับกลุ่มลูกค้าเสี่ยง (เช่น Churn) ให้ได้มากที่สุด การปรับลด threshold (เช่น $0.12$) จะช่วยเพิ่ม **Recall** ได้อย่างมาก แม้จะแลกมาด้วย Precision ที่ลดลง